# Experiment 4 - Comparative Study of Deep CNN Architectures Using Transfer Learning
CIFAR-10 transfer learning and fine-tuning with ImageNet-pretrained MobileNetV2.


In [ ]:
import time, json, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
classes=['Airplane','Automobile','Bird','Cat','Deer','Dog','Frog','Horse','Ship','Truck']
(x_all,y_all),(x_test,y_test)=keras.datasets.cifar10.load_data()
y_all=y_all.ravel(); y_test=y_test.ravel()
x_all=x_all.astype('float32')/255.0; x_test=x_test.astype('float32')/255.0
x_train,x_val,y_train,y_val=train_test_split(x_all,y_all,test_size=5000,random_state=SEED,stratify=y_all)
y_train_oh=keras.utils.to_categorical(y_train,10); y_val_oh=keras.utils.to_categorical(y_val,10); y_test_oh=keras.utils.to_categorical(y_test,10)
print('Train:',x_train.shape,'Validation:',x_val.shape,'Test:',x_test.shape,'Range:',float(x_train.min()),float(x_train.max()))
fig,axs=plt.subplots(2,5,figsize=(10,4.4))
for c,ax in enumerate(axs.ravel()):
    idx=np.flatnonzero(y_all==c)[0]; ax.imshow(x_all[idx]); ax.set_title(classes[c],fontsize=9); ax.axis('off')
fig.suptitle('CIFAR-10: One Sample from Each Class'); plt.tight_layout(rect=[0,0,1,.94]); plt.savefig(OUT/'sample_cifar10_images.png',dpi=220,bbox_inches='tight'); plt.show()
pre=keras.applications.mobilenet_v2.preprocess_input
base=keras.applications.MobileNetV2(weights='imagenet',include_top=False,input_shape=(32,32,3),pooling='avg'); base.trainable=False
def features(x,name):
    t=time.perf_counter(); z=base.predict(pre(x*255.0),batch_size=512,verbose=1); dt=time.perf_counter()-t; print(name,z.shape,f'{dt:.2f}s'); return z,dt
ftr,t1=features(x_train,'train'); fva,t2=features(x_val,'val'); fte,t3=features(x_test,'test'); feature_time=t1+t2+t3
def head(units=128,opt='Adam',lr=.001):
    m=keras.Sequential([layers.Input((ftr.shape[1],)),layers.Dense(units,activation='relu',name='dense'),layers.Dropout(.2),layers.Dense(10,activation='softmax',name='out')])
    o=keras.optimizers.Adam(lr) if opt=='Adam' else keras.optimizers.SGD(lr,momentum=.9)
    m.compile(optimizer=o,loss='categorical_crossentropy',metrics=['accuracy']); return m
hmodel=head(); t=time.perf_counter(); hist=hmodel.fit(ftr,y_train_oh,validation_data=(fva,y_val_oh),epochs=10,batch_size=32,verbose=2); head_time=time.perf_counter()-t
before_loss,before_acc=hmodel.evaluate(fte,y_test_oh,batch_size=512,verbose=0)
study=[{'Learning Rate':.001,'Batch Size':32,'Epochs':10,'Optimizer':'Adam','Dense Units':128,'Validation Accuracy':float(max(hist.history['val_accuracy']))}]
for lr,bs,ep,opt,u in [(.0001,32,10,'Adam',128),(.001,64,20,'Adam',256),(.001,32,10,'SGD',128)]:
    q=head(u,opt,lr); hh=q.fit(ftr,y_train_oh,validation_data=(fva,y_val_oh),epochs=ep,batch_size=bs,verbose=0); study.append({'Learning Rate':lr,'Batch Size':bs,'Epochs':ep,'Optimizer':opt,'Dense Units':u,'Validation Accuracy':float(max(hh.history['val_accuracy']))})
pd.DataFrame(study).sort_values('Validation Accuracy',ascending=False).to_csv(OUT/'hyperparameter_study.csv',index=False)
inp=keras.Input((32,32,3)); z=pre(inp*255.0); z=base(z,training=False); z=layers.Dense(128,activation='relu',name='dense')(z); z=layers.Dropout(.2)(z); out=layers.Dense(10,activation='softmax',name='out')(z); model=keras.Model(inp,out)
model.get_layer('dense').set_weights(hmodel.get_layer('dense').get_weights()); model.get_layer('out').set_weights(hmodel.get_layer('out').get_weights())
base.trainable=True
for l in base.layers[:-20]: l.trainable=False
for l in base.layers[-20:]:
    if isinstance(l,layers.BatchNormalization): l.trainable=False
model.compile(keras.optimizers.Adam(1e-5),loss='categorical_crossentropy',metrics=['accuracy'])
trainable_params=int(sum(np.prod(v.shape) for v in model.trainable_weights)); total_params=int(model.count_params())
t=time.perf_counter(); ft=model.fit(x_train,y_train_oh,validation_data=(x_val,y_val_oh),epochs=5,batch_size=256,verbose=2); ft_time=time.perf_counter()-t
epochs=np.arange(1,16); ta=hist.history['accuracy']+ft.history['accuracy']; va=hist.history['val_accuracy']+ft.history['val_accuracy']; tl=hist.history['loss']+ft.history['loss']; vl=hist.history['val_loss']+ft.history['val_loss']
def curve(vals,ylabel,title,fn):
    plt.figure(figsize=(7.5,4.2)); plt.plot(epochs,vals,marker='o',markersize=3); plt.axvline(10.5,ls='--',lw=1,label='Fine-tuning begins'); plt.xlabel('Epoch'); plt.ylabel(ylabel); plt.title(title); plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.savefig(OUT/fn,dpi=220,bbox_inches='tight'); plt.show()
curve(ta,'Accuracy','Training Accuracy vs Epoch','training_accuracy.png'); curve(va,'Accuracy','Validation Accuracy vs Epoch','validation_accuracy.png'); curve(tl,'Categorical Cross-Entropy Loss','Training Loss vs Epoch','training_loss.png'); curve(vl,'Categorical Cross-Entropy Loss','Validation Loss vs Epoch','validation_loss.png')
t=time.perf_counter(); final_loss,final_acc=model.evaluate(x_test,y_test_oh,batch_size=512,verbose=1); prob=model.predict(x_test,batch_size=512,verbose=1); pred_time=time.perf_counter()-t; yp=prob.argmax(1)
precision,recall,f1,_=precision_recall_fscore_support(y_test,yp,average='weighted',zero_division=0); cm=confusion_matrix(y_test,yp); rep=classification_report(y_test,yp,target_names=classes,output_dict=True,zero_division=0); pd.DataFrame(rep).T.to_csv(OUT/'classification_report.csv')
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cm,cmap='Blues'); fig.colorbar(im,ax=ax,fraction=.046,pad=.04); ax.set_xticks(range(10),labels=classes,rotation=45,ha='right'); ax.set_yticks(range(10),labels=classes); ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label'); ax.set_title('MobileNetV2 CIFAR-10 Confusion Matrix'); th=cm.max()/2
for i in range(10):
    for j in range(10): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=7,color='white' if cm[i,j]>th else 'black')
plt.tight_layout(); plt.savefig(OUT/'confusion_matrix.png',dpi=220,bbox_inches='tight'); plt.show()
metrics={'seed':SEED,'model':'MobileNetV2','pretraining':'ImageNet','training_images':len(x_train),'validation_images':len(x_val),'testing_images':len(x_test),'image_size':'32 x 32 x 3','classes':10,'optimizer':'Adam','initial_learning_rate':.001,'fine_tune_learning_rate':1e-5,'batch_size':32,'fine_tune_batch_size':256,'frozen_epochs':10,'fine_tune_epochs':5,'dense_units':128,'before_fine_tune_test_accuracy':float(before_acc),'after_fine_tune_test_accuracy':float(final_acc),'test_loss':float(final_loss),'precision_weighted':float(precision),'recall_weighted':float(recall),'f1_weighted':float(f1),'feature_extraction_time_seconds':feature_time,'classifier_training_time_seconds':head_time,'fine_tuning_time_seconds':ft_time,'training_time_seconds':head_time+ft_time,'end_to_end_compute_time_seconds':feature_time+head_time+ft_time,'prediction_time_seconds':pred_time,'total_parameters':total_params,'trainable_parameters_fine_tuning':trainable_params,'training_accuracy_history':[float(v) for v in ta],'validation_accuracy_history':[float(v) for v in va],'training_loss_history':[float(v) for v in tl],'validation_loss_history':[float(v) for v in vl],'confusion_matrix':cm.tolist()}
with open(OUT/'metrics.json','w') as f: json.dump(metrics,f,indent=2)
print(json.dumps({k:v for k,v in metrics.items() if not isinstance(v,list)},indent=2))
